In [ ]:
# ---------------- Imports configuration ----------------
from pathlib import Path
from collections import Counter, defaultdict
import ast
import json
import math
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
)
from sklearn.calibration import CalibratedClassifierCV

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    print("XGBoost unavailable:", e)

try:
    import lightgbm as lgb
    HAS_LGBM = True
except Exception as e:
    HAS_LGBM = False
    print("LightGBM unavailable:", e)

pd.set_option("display.max_columns", 200)

RANDOM_STATE = 42
N_SPLITS = 5
ID_COL = "person_id"       # use a de-identified study ID only; do not use MRN or direct identifiers
LABEL_COL = "label"
DAY_COL = "seq_day"
TOKEN_COL = "tokens"

# ---------------------------------------------------------------------
# Public-safe repository paths
# ---------------------------------------------------------------------
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUT_DIR = PROJECT_ROOT / "outputs" / "condition_lab_stdp_lag_tau_directed_both"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Expected de-identified input tables.
# Edit these filenames if your public/sample data use different names.
CONDITION_EVENT_DAY_PATH = DATA_DIR / "....csv"
CONDITION_EVENT_DAY_FALLBACKS = []

LAB_EVENT_DAY_PATH_MAIN = DATA_DIR / "....csv"
LAB_EVENT_DAY_PATH_SENS = DATA_DIR / "....csv"

# Choose lab file for this run. If the sensitivity file is unavailable, set this to False
# or point LAB_EVENT_DAY_PATH to an available de-identified lab event-day table.
USE_TUMOR_MARKER_LAB_FILE = True
LAB_EVENT_DAY_PATH = LAB_EVENT_DAY_PATH_SENS if USE_TUMOR_MARKER_LAB_FILE else LAB_EVENT_DAY_PATH_MAIN

# Optional raw numeric lab table. Leave as None unless rebuilding lab states directly from raw measurements.
RAW_LAB_TABLE_PATH = None

# Optional clinical covariate/label table. If labels are already present in event-day tables,
# this is still useful for cohort audits. Set USE_CLINICAL_COVARIATES=False if not available.
CLINICAL_COVARIATE_PATH = DATA_DIR / "....csv"
USE_CLINICAL_COVARIATES = True

# Public-safety output controls.
# Metrics and aggregate coefficients are safe to write. Patient-level OOF predictions contain study IDs
# and should not be committed publicly unless synthetic or explicitly approved for release.
SAVE_PATIENT_LEVEL_OUTPUTS = False
SAVE_COEFFICIENTS = True
SAVE_FOLD_INFO = True

# Transition settings.
PRIMARY_TRANSITION_MODES = ["forward"]
SENSITIVITY_TRANSITION_MODES = ["bidirectional"]

# Lag/tau grid for condition-lab and lab-condition transitions.
# Labs can change quickly, so short windows are prioritized.
LAG_TAU_GRID = [
    (1, 1),
    (2, 1),
    (3, 1),
    (3, 2),
    (7, 2),
    (7, 3),
    (14, 3),
    (14, 7),
    (30, 7),
]

# Primary lab representation levels.
LAB_LEVELS = ["labstate", "labaxis"]

# Primary lab states for biologic transition features. Use abnormal-only for causal/trajectory-like analyses.
LAB_STATE_FILTERS = ["abnormal"]
LAB_STATE_FILTER = LAB_STATE_FILTERS[0] if isinstance(LAB_STATE_FILTERS, (list, tuple)) else LAB_STATE_FILTERS

# The de-identified lab files are expected to be collapsed event-day tables. Set to event_day.
LAB_INPUT_MODE = "event_day"  # "auto", "event_day", or "raw"
INCLUDE_NORMAL_LAB_STATES = False

TOP_K_FREQ = 1000
TOP_K_TRANSITION = 2500
TOP_K_COMBINED = 3500
MIN_PATIENT_FREQ = 5
SPEC_TARGETS = [0.80, 0.85, 0.90, 0.95]

RUN_PREDICTION_HEAD_TUNING = True
RUN_TREE_HEADS = True
INNER_CV_SPLITS = 3

# Utility: choose first existing file among preferred path + fallbacks.
def choose_existing_path(preferred_path, fallback_paths=None, description="input file"):
    paths = [preferred_path] + list(fallback_paths or [])
    for p in paths:
        if p is not None and Path(p).exists():
            if Path(p) != Path(preferred_path):
                print(f"Using fallback {description}: {p}")
            return Path(p)
    print(f"No existing {description} found. Checked:")
    for p in paths:
        print("  -", p)
    # Return preferred path so downstream error message is explicit.
    return Path(preferred_path)

CONDITION_EVENT_DAY_PATH = choose_existing_path(
    CONDITION_EVENT_DAY_PATH,
    CONDITION_EVENT_DAY_FALLBACKS,
    description="condition event-day table",
)

print("PUBLIC-SAFE NOTE: use only de-identified input files and do not commit patient-level outputs.")
print("Project root:", PROJECT_ROOT)
print("Output directory:", OUT_DIR)
print("Condition event-day file:", CONDITION_EVENT_DAY_PATH)
print("Lab event-day file:", LAB_EVENT_DAY_PATH)
print("Clinical covariate file:", CLINICAL_COVARIATE_PATH)
print("Use tumor-marker lab file:", USE_TUMOR_MARKER_LAB_FILE)
print("Save patient-level outputs:", SAVE_PATIENT_LEVEL_OUTPUTS)


In [ ]:
# ---------------- Public file audit ----------------
input_files = {
    "condition_event_day": CONDITION_EVENT_DAY_PATH,
    "lab_event_day_main": LAB_EVENT_DAY_PATH_MAIN,
    "lab_event_day_sensitivity_plus_tumor_markers": LAB_EVENT_DAY_PATH_SENS,
    "lab_event_day_used": LAB_EVENT_DAY_PATH,
    "clinical_covariates": CLINICAL_COVARIATE_PATH if USE_CLINICAL_COVARIATES else None,
}

file_audit = []
for name, path in input_files.items():
    p = Path(path) if path is not None else None
    file_audit.append({
        "name": name,
        "path": str(p) if p is not None else None,
        "exists": bool(p.exists()) if p is not None else False,
    })
file_audit_df = pd.DataFrame(file_audit)
display(file_audit_df)

missing = file_audit_df.loc[~file_audit_df["exists"], "name"].tolist()
required_missing = [x for x in missing if x in {"condition_event_day", "lab_event_day_used"}]
if required_missing:
    raise FileNotFoundError(
        "Required de-identified input files are missing: "
        f"{required_missing}. Place CSVs under data/ or edit the configuration cell."
    )


In [ ]:
# ---------------- Robust event-day and raw-lab loaders ----------------
DAY_ALIASES = ["seq_day", "event_day", "days_from_surgery", "day", "days_to_surgery", "relative_day"]
TOKEN_ALIASES = ["tokens", "token", "condition_token", "lab_tokens", "lab_token", "feature", "event_node"]
VALUE_ALIASES = ["value_as_number", "numeric_value", "lab_value", "result_value", "value", "result_num"]
LAB_NAME_ALIASES = ["lab_name", "measurement_name", "concept_name", "measurement_concept_name", "test_name", "component_name", "variable", "loinc_name"]
LAB_CODE_ALIASES = ["loinc_code", "loinc", "concept_code", "measurement_source_value", "measurement_source_concept_code", "source_code", "lab_code"]
REF_LOW_ALIASES = ["range_low", "ref_low", "reference_low", "low_normal", "lower_limit", "range_low_value"]
REF_HIGH_ALIASES = ["range_high", "ref_high", "reference_high", "high_normal", "upper_limit", "range_high_value"]
UNIT_ALIASES = ["unit", "unit_source_value", "unit_concept_name", "unit_name"]
FLAG_ALIASES = ["range_flag", "abnormal_flag", "flag", "result_flag", "interpretation"]
BAD_NAME_REGEX = re.compile(r"(metrics|prediction|coefficient|importance|manifest|summary|audit|gap|inter_event)", re.I)


def parse_token_payload(x):
    """Return a clean list of tokens from scalar, list-like string, or delimited payload."""
    if isinstance(x, (list, tuple, set, np.ndarray)):
        vals = list(x)
    elif pd.isna(x):
        vals = []
    else:
        s = str(x).strip()
        vals = None
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                vals = parsed if isinstance(parsed, (list, tuple, set)) else [parsed]
            except Exception:
                vals = None
        if vals is None:
            if "|" in s:
                vals = s.split("|")
            elif ";" in s:
                vals = s.split(";")
            else:
                vals = [s]
    return sorted(set(str(v).strip() for v in vals if str(v).strip() and str(v).strip().lower() != "nan"))


def find_existing_or_raise(path, description):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"Missing {description}: {p}. Place a de-identified CSV at this path or edit the config cell."
        )
    return p


def _first_existing_col(df, aliases):
    lower = {str(c).lower(): c for c in df.columns}
    return next((lower[a] for a in aliases if a in lower), None)


def load_event_day_table(path, kind="condition"):
    path = find_existing_or_raise(path, f"{kind} event-day table")
    df = pd.read_csv(path, low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]
    lower = {c.lower(): c for c in df.columns}

    id_col = lower.get(ID_COL, ID_COL)
    day_col = next((lower[c] for c in DAY_ALIASES if c in lower), None)
    tok_col = next((lower[c] for c in TOKEN_ALIASES if c in lower), None)

    if id_col not in df.columns or day_col is None or tok_col is None:
        raise ValueError(
            f"{kind} table must include person_id, a relative-day column, and a token column. Columns: {list(df.columns)}"
        )

    out = df.rename(columns={id_col: ID_COL, day_col: DAY_COL, tok_col: TOKEN_COL}).copy()
    out[ID_COL] = pd.to_numeric(out[ID_COL], errors="coerce")
    out[DAY_COL] = pd.to_numeric(out[DAY_COL], errors="coerce")
    out = out.dropna(subset=[ID_COL, DAY_COL]).copy()
    out[ID_COL] = out[ID_COL].astype(int)
    out[DAY_COL] = out[DAY_COL].astype(int)
    out[TOKEN_COL] = out[TOKEN_COL].apply(parse_token_payload)
    out = out[out[TOKEN_COL].map(len) > 0].copy()

    collapsed = (
        out.groupby([ID_COL, DAY_COL], as_index=False)[TOKEN_COL]
        .agg(lambda xs: sorted(set(t for sub in xs for t in sub)))
    )

    for optional in [LABEL_COL, "cohort_name"]:
        if optional in out.columns:
            collapsed = collapsed.merge(
                out[[ID_COL, optional]].dropna().drop_duplicates(ID_COL),
                on=ID_COL,
                how="left",
            )

    print(f"{kind} event-day table:", collapsed.shape, "patients:", collapsed[ID_COL].nunique())
    return collapsed


def load_clinical_covariates(path):
    p = Path(path)
    if not p.exists():
        print("No clinical covariate table found. Proceeding with labels from event-day tables.")
        return pd.DataFrame(columns=[ID_COL])
    df = pd.read_csv(p, low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]
    print("Clinical covariates:", df.shape)
    return df


## Lab abnormality-state construction

For cross-modal condition-lab transitions, laboratory measurements are converted into categorical abnormality states before feature construction. The notebook uses local reference ranges when available and prespecified adult fallback thresholds when local ranges are missing. These thresholds are intended for feature engineering and physiologic-axis grouping, not for clinical diagnosis.

Primary transition models use abnormal lab states only. Normal lab measurements are treated as documentation/utilization information and are adjusted for separately through utilization features rather than used as primary biological transition nodes.


In [ ]:
# ---------------- Literature-grounded lab-state definitions and biologic-axis mapping ----------------
# Laboratory events are value-based states, not just measurement mentions.
# Priority: local reference range > structured high/low flag > adult fallback threshold.
# Fallback thresholds are for feature engineering/physiologic-axis grouping, not clinical diagnosis.

LAB_STATE_DEFINITIONS = {
    # variable_key: axis, fallback low/high thresholds, states to emit
    "ca19_9": {
        "axis": "tumor_marker",
        "fallback_high": 37.0,
        "high_state": "ca199_high",
        "rationale": "PDAC tumor marker; interpret cautiously with biliary obstruction/cholangitis/Lewis antigen status.",
    },
    "bilirubin_total": {
        "axis": "cholestasis_hepatobiliary",
        "fallback_high": 1.2,
        "high_state": "bilirubin_high",
        "rationale": "Cholestasis/hepatobiliary dysfunction; relevant to PDAC obstruction and CA19-9 interpretation.",
    },
    "alkaline_phosphatase": {
        "axis": "cholestasis_hepatobiliary",
        "high_state": "alk_phos_high",
        "rationale": "Cholestatic liver/biliary injury; use local reference range because intervals vary.",
    },
    "ast": {
        "axis": "liver_injury",
        "high_state": "ast_high",
        "rationale": "Hepatocellular injury/systemic illness axis; local reference range preferred.",
    },
    "alt": {
        "axis": "liver_injury",
        "high_state": "alt_high",
        "rationale": "Hepatocellular injury/systemic illness axis; local reference range preferred.",
    },
    "albumin": {
        "axis": "nutrition_inflammation",
        "fallback_low": 3.5,
        "low_state": "albumin_low",
        "rationale": "Nutritional/inflammatory burden.",
    },
    "wbc": {
        "axis": "inflammation",
        "fallback_high": 11.0,
        "high_state": "wbc_high",
        "rationale": "Leukocytosis/inflammatory or infectious stress.",
    },
    "neutrophil": {
        "axis": "inflammation",
        "high_state": "neutrophil_high",
        "rationale": "Neutrophil-predominant inflammation; local reference range preferred.",
    },
    "lymphocyte": {
        "axis": "immune_inflammation",
        "low_state": "lymphocyte_low",
        "rationale": "Lymphopenia/inflammatory-nutritional axis; local reference range preferred.",
    },
    "hemoglobin": {
        "axis": "hematologic",
        "fallback_low": 12.0,
        "low_state": "hemoglobin_low",
        "rationale": "Anemia/systemic illness burden; sex-specific or local reference ranges preferred.",
    },
    "platelet": {
        "axis": "hematologic_inflammation",
        "fallback_low": 150.0,
        "fallback_high": 450.0,
        "low_state": "platelet_low",
        "high_state": "platelet_high",
        "rationale": "Thrombocytopenia/thrombocytosis, inflammation or illness severity.",
    },
    "glucose": {
        "axis": "glycemic_metabolic",
        "fallback_high": 140.0,
        "high_state": "glucose_high",
        "rationale": "Hyperglycemia/diabetes-related metabolic dysregulation; random glucose fallback.",
    },
    "creatinine": {
        "axis": "renal",
        "fallback_high": 1.2,
        "high_state": "creatinine_high",
        "rationale": "Renal/systemic illness axis.",
    },
    "bun": {
        "axis": "renal",
        "fallback_high": 23.0,
        "high_state": "bun_high",
        "rationale": "Azotemia, dehydration, renal/catabolic systemic illness axis.",
    },
    "sodium": {
        "axis": "electrolyte",
        "fallback_low": 135.0,
        "fallback_high": 145.0,
        "low_state": "sodium_low",
        "high_state": "sodium_high",
        "rationale": "Fluid/electrolyte disturbance.",
    },
    "potassium": {
        "axis": "electrolyte",
        "fallback_low": 3.5,
        "fallback_high": 5.0,
        "low_state": "potassium_low",
        "high_state": "potassium_high",
        "rationale": "Renal function, intake, fluid shifts, medications, systemic illness.",
    },
}

# Synonyms and optional LOINC/code hints. Add site-specific names here after manual audit.
LAB_SYNONYMS = {
    "ca19_9": ["ca 19", "ca19", "ca_19", "carbohydrate antigen 19", "cancer antigen 19"],
    "bilirubin_total": ["total bilirubin", "bilirubin total", "bilirubin", "tbil", "bili total"],
    "alkaline_phosphatase": ["alkaline phosphatase", "alk phos", "alp"],
    "ast": ["aspartate aminotransferase", "ast", "sgot"],
    "alt": ["alanine aminotransferase", "alt", "sgpt"],
    "albumin": ["albumin"],
    "wbc": ["white blood", "wbc", "leukocyte"],
    "neutrophil": ["neutrophil", "neutrophils", "anc", "absolute neutrophil"],
    "lymphocyte": ["lymphocyte", "lymphocytes", "absolute lymph"],
    "hemoglobin": ["hemoglobin", "hgb", "hb"],
    "platelet": ["platelet", "plt"],
    "glucose": ["glucose", "blood sugar"],
    "creatinine": ["creatinine", "serum creatinine"],
    "bun": ["blood urea nitrogen", "bun", "urea nitrogen"],
    "sodium": ["sodium", "na"],
    "potassium": ["potassium", "k"],
}

LAB_CODE_HINTS = {
    # These are optional hints only; local concept names/reference ranges should be preferred.
    # Add institution-specific LOINC/code mappings here if needed.
}

LAB_AXIS_MAP = {k: v["axis"] for k, v in LAB_STATE_DEFINITIONS.items()}
# Additional non-primary labs accepted if already tokenized upstream.
LAB_AXIS_MAP.update({
    "bilirubin_direct": "cholestasis_hepatobiliary",
    "ggt": "cholestasis_hepatobiliary",
    "prealbumin": "nutrition_inflammation",
    "total_protein": "nutrition_inflammation",
    "crp": "inflammation",
    "esr": "inflammation",
    "hematocrit": "hematologic",
    "inr": "coagulation",
    "pt": "coagulation",
    "ptt": "coagulation",
    "egfr": "renal",
    "bicarbonate": "electrolyte_acid_base",
    "chloride": "electrolyte",
    "calcium": "electrolyte",
    "magnesium": "electrolyte",
    "phosphorus": "electrolyte",
    "hba1c": "glycemic_metabolic",
    "a1c": "glycemic_metabolic",
    "bmi": "nutrition_anthropometric",
    "weight": "nutrition_anthropometric",
})

ABNORMAL_STATES = {"high", "low", "abnormal", "worse", "rising", "falling", "critical_high", "critical_low"}
NORMAL_STATES = {"normal", "stable", "observed"}


def normalize_label(s):
    s = str(s).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    s = re.sub(r"_+", "_", s)
    return s


def standardize_lab_variable(name=None, code=None):
    """Map raw lab name/code to a primary lab variable key."""
    code_s = normalize_label(code) if code is not None and not pd.isna(code) else ""
    name_s = normalize_label(name) if name is not None and not pd.isna(name) else ""
    combined = f"{name_s} {code_s}".strip()

    if code_s in LAB_CODE_HINTS:
        return LAB_CODE_HINTS[code_s]

    # Prefer longer/more specific synonyms first.
    for variable, synonyms in LAB_SYNONYMS.items():
        for syn in sorted(synonyms, key=len, reverse=True):
            syn_n = normalize_label(syn)
            if re.search(rf"(^|_){re.escape(syn_n)}(_|$)", combined):
                return variable
            if syn_n in combined and len(syn_n) >= 4:
                return variable
    return None


def _to_float(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(",", "")
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    return float(m.group(0)) if m else np.nan


def flag_to_state(flag):
    if pd.isna(flag):
        return None
    s = str(flag).strip().lower()
    if s in {"h", "hi", "high", "above", "abnormally high", ">"}:
        return "high"
    if s in {"l", "lo", "low", "below", "abnormally low", "<"}:
        return "low"
    if "critical" in s and "high" in s:
        return "critical_high"
    if "critical" in s and "low" in s:
        return "critical_low"
    return None


def categorize_lab_value(variable, value, ref_low=np.nan, ref_high=np.nan, flag=None):
    """Return high/low/normal/None for a standardized variable using local range first, then fallback."""
    if variable not in LAB_STATE_DEFINITIONS:
        return None
    d = LAB_STATE_DEFINITIONS[variable]
    value = _to_float(value)
    ref_low = _to_float(ref_low)
    ref_high = _to_float(ref_high)

    # Structured flag can identify abnormality when numeric reference range is unavailable.
    flagged = flag_to_state(flag)

    if not pd.isna(value):
        if not pd.isna(ref_high) and value > ref_high:
            return "high"
        if not pd.isna(ref_low) and value < ref_low:
            return "low"
        if pd.isna(ref_high) and "fallback_high" in d and value > d["fallback_high"]:
            return "high"
        if pd.isna(ref_low) and "fallback_low" in d and value < d["fallback_low"]:
            return "low"
        if INCLUDE_NORMAL_LAB_STATES:
            return "normal"

    if flagged in {"high", "critical_high"} and "high_state" in d:
        return flagged
    if flagged in {"low", "critical_low"} and "low_state" in d:
        return flagged
    return None


def lab_state_name(variable, state):
    """Map generic high/low state to manuscript-style abnormality token."""
    d = LAB_STATE_DEFINITIONS.get(variable, {})
    state = normalize_label(state)
    if state in {"high", "critical_high"} and "high_state" in d:
        return d["high_state"] if state == "high" else d["high_state"].replace("_high", "_critical_high")
    if state in {"low", "critical_low"} and "low_state" in d:
        return d["low_state"] if state == "low" else d["low_state"].replace("_low", "_critical_low")
    if state == "normal":
        return f"{variable}_normal"
    return f"{variable}_{state}"


def make_value_based_lab_tokens(variable, state, level="labstate"):
    variable = normalize_label(variable)
    state = normalize_label(state)
    axis = LAB_AXIS_MAP.get(variable, "other_lab_axis")
    state_name = lab_state_name(variable, state)
    if level == "labstate":
        return f"labstate::{state_name}"
    if level == "labaxis":
        # Use axis plus high/low direction for biological interpretability.
        if state in {"critical_high"}:
            direction = "high"
        elif state in {"critical_low"}:
            direction = "low"
        elif state in {"high", "low", "normal"}:
            direction = state
        else:
            direction = "abnormal" if state in ABNORMAL_STATES else state
        return f"labaxis::{axis}::{direction}"
    if level == "labaxis_binary_abnormal":
        binary = "abnormal" if state in ABNORMAL_STATES else "not_abnormal"
        return f"labaxis::{axis}::{binary}"
    raise ValueError(f"Unknown lab level: {level}")


def parse_lab_token(tok):
    """Return lab_type/state from heterogeneous pre-tokenized lab strings."""
    raw = str(tok).strip()
    s = raw.lower().replace("::", ":")
    parts = [normalize_label(p) for p in s.split(":") if p.strip()]

    if len(parts) >= 3 and parts[0] in {"lab", "labtraj", "labstate", "tumor_marker"}:
        # Support labstate::albumin::low and labstate::albumin_low.
        if len(parts) >= 3:
            return parts[1], parts[2]

    cleaned = normalize_label(raw)
    for suffix in ["critical_high", "critical_low", "abnormal", "rising", "falling", "worse", "high", "low", "normal", "stable", "observed"]:
        suf = "_" + suffix
        if cleaned.endswith(suf):
            lab_type = cleaned[: -len(suf)]
            # If this is already a manuscript state, map back approximately.
            for var, d in LAB_STATE_DEFINITIONS.items():
                if d.get("high_state") == cleaned or d.get("low_state") == cleaned:
                    return var, suffix
            return lab_type, suffix

    return cleaned, "observed"


def lab_axis_for_type(lab_type):
    lab_type = normalize_label(lab_type)
    if lab_type in LAB_AXIS_MAP:
        return LAB_AXIS_MAP[lab_type]
    for key, axis in LAB_AXIS_MAP.items():
        if key in lab_type or lab_type in key:
            return axis
    return "other_lab_axis"


def make_lab_token(lab_type, state, level="labstate"):
    # If lab_type is a raw variable, use value-based token naming.
    var = standardize_lab_variable(lab_type, None) or normalize_label(lab_type)
    state = normalize_label(state)
    if var in LAB_STATE_DEFINITIONS and state in {"high", "low", "critical_high", "critical_low", "normal"}:
        return make_value_based_lab_tokens(var, state, level=level)

    axis = lab_axis_for_type(var)
    if level == "labstate":
        return f"labstate::{var}::{state}"
    if level == "labaxis":
        return f"labaxis::{axis}::{state}"
    if level == "labaxis_binary_abnormal":
        binary = "abnormal" if state in ABNORMAL_STATES else "not_abnormal"
        return f"labaxis::{axis}::{binary}"
    raise ValueError(f"Unknown lab level: {level}")


def lab_token_passes_filter(lab_type, state, lab_state_filter="abnormal"):
    state = normalize_label(state)
    if lab_state_filter in [None, "all"]:
        return True
    if lab_state_filter == "abnormal":
        return state in ABNORMAL_STATES or state in {"high", "low", "critical_high", "critical_low"}
    if lab_state_filter == "normal":
        return state in NORMAL_STATES
    if lab_state_filter == "observed_or_abnormal":
        return state in ABNORMAL_STATES or state in NORMAL_STATES
    raise ValueError(f"Unknown lab_state_filter: {lab_state_filter}")


def remap_lab_event_days(lab_days, level="labstate", lab_state_filter="abnormal"):
    rows = []
    for _, r in lab_days.iterrows():
        toks = []
        for tok in r[TOKEN_COL]:
            lt, st = parse_lab_token(tok)
            if lab_token_passes_filter(lt, st, lab_state_filter=lab_state_filter):
                toks.append(make_lab_token(lt, st, level=level))
        if toks:
            row = {ID_COL: int(r[ID_COL]), DAY_COL: int(r[DAY_COL]), TOKEN_COL: sorted(set(toks))}
            for c in [LABEL_COL, "cohort_name"]:
                if c in lab_days.columns:
                    row[c] = r[c]
            rows.append(row)
    out = pd.DataFrame(rows)
    if len(out) == 0:
        return pd.DataFrame(columns=[ID_COL, DAY_COL, TOKEN_COL, LABEL_COL, "cohort_name"])
    out = (
        out.groupby([ID_COL, DAY_COL], as_index=False)[TOKEN_COL]
        .agg(lambda xs: sorted(set(t for sub in xs for t in sub)))
    ).merge(
        out[[ID_COL] + [c for c in [LABEL_COL, "cohort_name"] if c in out.columns]].drop_duplicates(ID_COL),
        on=ID_COL,
        how="left",
    )
    return out


def build_lab_event_days_from_raw(path):
    """Build abnormal lab event-days from raw numeric lab rows using local ranges and fallback thresholds."""
    path = find_existing_or_raise(path, "raw lab measurement table")
    df = pd.read_csv(path, low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]

    id_col = _first_existing_col(df, [ID_COL, "patient_id", "person_id"])
    day_col = _first_existing_col(df, DAY_ALIASES)
    value_col = _first_existing_col(df, VALUE_ALIASES)
    name_col = _first_existing_col(df, LAB_NAME_ALIASES)
    code_col = _first_existing_col(df, LAB_CODE_ALIASES)
    low_col = _first_existing_col(df, REF_LOW_ALIASES)
    high_col = _first_existing_col(df, REF_HIGH_ALIASES)
    flag_col = _first_existing_col(df, FLAG_ALIASES)

    required_missing = [label for label, col in [("patient id", id_col), ("relative day", day_col), ("numeric value", value_col)] if col is None]
    if required_missing:
        raise ValueError(f"Raw lab table missing required columns: {required_missing}. Columns: {list(df.columns)}")
    if name_col is None and code_col is None:
        raise ValueError("Raw lab table needs at least one lab name or lab code column.")

    work = df.copy()
    work[ID_COL] = pd.to_numeric(work[id_col], errors="coerce")
    work[DAY_COL] = pd.to_numeric(work[day_col], errors="coerce")
    work["_value"] = work[value_col].apply(_to_float)
    work = work.dropna(subset=[ID_COL, DAY_COL, "_value"]).copy()
    work[ID_COL] = work[ID_COL].astype(int)
    work[DAY_COL] = work[DAY_COL].astype(int)

    work["_lab_variable"] = [
        standardize_lab_variable(work.at[i, name_col] if name_col else None, work.at[i, code_col] if code_col else None)
        for i in work.index
    ]

    def _row_state(r):
        return categorize_lab_value(
            r["_lab_variable"],
            r["_value"],
            r[low_col] if low_col else np.nan,
            r[high_col] if high_col else np.nan,
            r[flag_col] if flag_col else None,
        )

    work["_state"] = work.apply(_row_state, axis=1)
    before = len(work)
    work = work[work["_lab_variable"].notna() & work["_state"].notna()].copy()
    print("Raw lab rows parsed:", before, "-> value-based lab-state rows:", len(work))
    print("Patients with lab states:", work[ID_COL].nunique())

    work["_token"] = [make_value_based_lab_tokens(v, s, level="labstate") for v, s in zip(work["_lab_variable"], work["_state"])]

    out = (
        work.groupby([ID_COL, DAY_COL], as_index=False)["_token"]
        .agg(lambda xs: sorted(set(xs)))
        .rename(columns={"_token": TOKEN_COL})
    )

    for optional in [LABEL_COL, "cohort_name"]:
        if optional in work.columns:
            out = out.merge(work[[ID_COL, optional]].dropna().drop_duplicates(ID_COL), on=ID_COL, how="left")

    print("Lab event-day table from raw labs:", out.shape, "patients:", out[ID_COL].nunique())
    return out


def load_lab_input_event_days():
    """Load lab event-days directly or build them from raw labs depending on available files/config."""
    mode = LAB_INPUT_MODE.lower()
    if mode == "event_day":
        return load_event_day_table(LAB_EVENT_DAY_PATH, kind="lab")
    if mode == "raw":
        return build_lab_event_days_from_raw(RAW_LAB_TABLE_PATH)
    # auto
    if Path(LAB_EVENT_DAY_PATH).exists():
        return load_event_day_table(LAB_EVENT_DAY_PATH, kind="lab")
    return build_lab_event_days_from_raw(RAW_LAB_TABLE_PATH)


# Display the lab-state table used by this notebook.
lab_state_table = pd.DataFrame([
    {
        "lab_variable": k,
        "axis": v.get("axis"),
        "low_state": v.get("low_state"),
        "fallback_low": v.get("fallback_low"),
        "high_state": v.get("high_state"),
        "fallback_high": v.get("fallback_high"),
        "rationale": v.get("rationale"),
    }
    for k, v in LAB_STATE_DEFINITIONS.items()
])
display(lab_state_table)


In [ ]:
# ---------------- Load condition and lab event-day tables ----------------
condition_days = load_event_day_table(CONDITION_EVENT_DAY_PATH, kind="condition")
lab_days_raw = load_lab_input_event_days()
clinical_df = load_clinical_covariates(CLINICAL_COVARIATE_PATH) if USE_CLINICAL_COVARIATES else pd.DataFrame(columns=[ID_COL])

print("Condition event-days:", condition_days.shape, "patients:", condition_days[ID_COL].nunique())
print("Raw/prebuilt lab event-days:", lab_days_raw.shape, "patients:", lab_days_raw[ID_COL].nunique())
display(lab_days_raw.head())

# Build the lab representations used downstream.
# labstate keeps specific value-based abnormalities, e.g. labstate::bilirubin_high.
# labaxis maps those states to broader biology axes, e.g. labaxis::cholestasis_hepatobiliary::high.
lab_days_by_level = {}
for level in LAB_LEVELS:
    remapped = remap_lab_event_days(
        lab_days_raw,
        level=level,
        lab_state_filter=globals().get("LAB_STATE_FILTER", "abnormal"),
    )
    lab_days_by_level[level] = remapped
    print(f"Lab event-days [{level}]:", remapped.shape, "patients:", remapped[ID_COL].nunique())
    display(remapped.head())

# Fail early if no lab-state representation was created. This usually means the lab token parser
# does not recognize the tokens in the input lab event-day file, or the raw-lab abnormality mapping
# did not emit abnormal states.
if all(len(df) == 0 for df in lab_days_by_level.values()):
    print("WARNING: No remapped lab event-days were created. Inspect lab_days_raw tokens below:")
    display(lab_days_raw[[ID_COL, DAY_COL, TOKEN_COL]].head(20))


In [ ]:
# ---------------- Outcome table and cohort preservation ----------------
def build_y_df(condition_days, lab_days_by_level, clinical_df):
    parts = []
    if LABEL_COL in condition_days.columns:
        parts.append(condition_days[[ID_COL, LABEL_COL]].dropna())
    for df in lab_days_by_level.values():
        if LABEL_COL in df.columns:
            parts.append(df[[ID_COL, LABEL_COL]].dropna())
    if LABEL_COL in clinical_df.columns:
        parts.append(clinical_df[[ID_COL, LABEL_COL]].dropna())
    if not parts:
        for alt in ["early_recurrence", "recurrence_6m", "case", "outcome"]:
            if alt in clinical_df.columns:
                parts.append(clinical_df[[ID_COL, alt]].rename(columns={alt: LABEL_COL}).dropna())
                break
    if not parts:
        raise ValueError("Cannot find outcome label column in event-day or clinical tables.")

    y = pd.concat(parts, ignore_index=True)
    y[ID_COL] = pd.to_numeric(y[ID_COL], errors="coerce")
    y[LABEL_COL] = pd.to_numeric(y[LABEL_COL], errors="coerce")
    y = y.dropna(subset=[ID_COL, LABEL_COL]).copy()
    y[ID_COL] = y[ID_COL].astype(int)
    y[LABEL_COL] = y[LABEL_COL].astype(int)
    y = y.drop_duplicates(ID_COL)

    cohort_parts = []
    if "cohort_name" in condition_days.columns:
        cohort_parts.append(condition_days[[ID_COL, "cohort_name"]].dropna())
    for df in lab_days_by_level.values():
        if "cohort_name" in df.columns:
            cohort_parts.append(df[[ID_COL, "cohort_name"]].dropna())
    if "cohort_name" in clinical_df.columns:
        cohort_parts.append(clinical_df[[ID_COL, "cohort_name"]].dropna())
    if cohort_parts:
        cohort_df = pd.concat(cohort_parts).drop_duplicates(ID_COL)
        y = y.merge(cohort_df, on=ID_COL, how="left")
    else:
        y["cohort_name"] = ""
    return y

# Source cohort = patients with condition events or remapped lab events and a known label.
source_ids = set(condition_days[ID_COL].astype(int))
for df in lab_days_by_level.values():
    source_ids |= set(df[ID_COL].astype(int))

y_df = build_y_df(condition_days, lab_days_by_level, clinical_df)
y_df = y_df[y_df[ID_COL].isin(source_ids)].copy()
y_df = y_df.sort_values(ID_COL).reset_index(drop=True)
all_patient_ids = y_df[ID_COL].astype(int).tolist()

print("Outcome table:", y_df.shape, "patients:", y_df[ID_COL].nunique(), "case_rate:", y_df[LABEL_COL].mean())
display(y_df[LABEL_COL].value_counts().sort_index().rename_axis("label").reset_index(name="n_patients"))

assert y_df[ID_COL].is_unique, "y_df must have one row per patient."
assert set(all_patient_ids).issubset(source_ids), "y_df contains patients not in source event tables."


In [ ]:

# ---------------- Cohort initialization helper ----------------
def ensure_patient_cohort_initialized():
    """Ensure y_df and all_patient_ids exist before feature construction.

    This makes the notebook safer if a downstream cell is run manually before the
    outcome/cohort cell. The intended run mode is still Kernel -> Restart & Run All.
    """
    global y_df, all_patient_ids

    if "all_patient_ids" in globals() and "y_df" in globals():
        return

    if "build_y_df" not in globals():
        raise NameError(
            "build_y_df is not defined. Run the outcome table/cohort preservation cell first."
        )
    if "condition_days" not in globals() or "lab_days_by_level" not in globals() or "clinical_df" not in globals():
        raise NameError(
            "condition_days, lab_days_by_level, or clinical_df is not defined. Run the loader and lab-remapping cells first."
        )

    source_ids = set(condition_days[ID_COL].astype(int)) if len(condition_days) else set()
    for _df in lab_days_by_level.values():
        if _df is not None and len(_df):
            source_ids |= set(_df[ID_COL].astype(int))

    y_df = build_y_df(condition_days, lab_days_by_level, clinical_df)
    y_df = y_df[y_df[ID_COL].isin(source_ids)].copy()
    y_df = y_df.sort_values(ID_COL).reset_index(drop=True)
    all_patient_ids = y_df[ID_COL].astype(int).tolist()

    print("Initialized patient cohort:", len(all_patient_ids), "patients")
    assert y_df[ID_COL].is_unique, "y_df must have one row per patient."
    assert set(all_patient_ids).issubset(source_ids), "y_df contains patients not in source event tables."


In [ ]:
# ---------------- Sequence and feature builders ----------------
def table_to_sequences(event_df):
    seqs = {}
    if event_df is None or len(event_df) == 0:
        return seqs
    for pid, g in event_df.groupby(ID_COL):
        rows = []
        for _, r in g.sort_values(DAY_COL).iterrows():
            toks = sorted(set(str(t).strip() for t in r[TOKEN_COL] if str(t).strip()))
            if toks:
                rows.append((int(r[DAY_COL]), toks))
        seqs[int(pid)] = rows
    return seqs

ensure_patient_cohort_initialized()

condition_seq = table_to_sequences(condition_days)
lab_seq_by_level = {key: table_to_sequences(df) for key, df in lab_days_by_level.items()}


def build_frequency_dicts(seqs, prefix):
    rows = []
    for pid in all_patient_ids:
        d = {ID_COL: int(pid)}
        cnt = Counter()
        for day, toks in seqs.get(int(pid), []):
            for tok in toks:
                cnt[f"{prefix}::{tok}"] += 1
        for k, v in cnt.items():
            d[k] = math.log1p(min(v, 20))
        rows.append(d)
    return rows


def build_cross_transition_dicts(
    source_seqs,
    target_seqs,
    source_prefix,
    target_prefix,
    direction_name,
    max_lag,
    tau,
    mode="forward",
    include_same_day=False,
):
    """Build condition-lab or lab-condition transition features.

    mode='forward': source event must occur before target event.
    mode='bidirectional': source and target are paired if abs(day gap) <= max_lag, ignoring order.
    """
    rows = []
    for pid in all_patient_ids:
        d = {ID_COL: int(pid)}
        src_events = [(day, tok) for day, toks in source_seqs.get(int(pid), []) for tok in toks]
        tgt_events = [(day, tok) for day, toks in target_seqs.get(int(pid), []) for tok in toks]
        src_events = sorted(src_events)
        tgt_events = sorted(tgt_events)
        trans = Counter()

        if mode == "forward":
            for d1, a in src_events:
                for d2, b in tgt_events:
                    gap = d2 - d1
                    if gap < 0:
                        continue
                    if gap == 0 and not include_same_day:
                        continue
                    if gap > max_lag:
                        continue
                    w = math.exp(-gap / max(float(tau), 1e-9))
                    key = f"{direction_name}_forward_lag{max_lag}_tau{tau}::{source_prefix}:{a}->{target_prefix}:{b}"
                    trans[key] += w

        elif mode == "bidirectional":
            for d1, a in src_events:
                for d2, b in tgt_events:
                    gap = abs(d2 - d1)
                    if gap == 0 and not include_same_day:
                        # Same-day can be enabled separately. Default is false to reduce co-documentation dominance.
                        continue
                    if gap > max_lag:
                        continue
                    w = math.exp(-gap / max(float(tau), 1e-9))
                    # Symmetric condition-lab proximity key.
                    key = f"{direction_name}_bidirectional_lag{max_lag}_tau{tau}::{source_prefix}:{a}<->{target_prefix}:{b}"
                    trans[key] += w
        else:
            raise ValueError(f"Unknown mode: {mode}")

        for k, v in trans.items():
            d[k] = v
        rows.append(d)
    return rows


def merge_dicts(name, dict_lists):
    by_pid = {int(pid): {ID_COL: int(pid)} for pid in all_patient_ids}
    for dicts in dict_lists:
        for d in dicts:
            pid = int(d[ID_COL])
            if pid not in by_pid:
                continue
            for k, v in d.items():
                if k == ID_COL:
                    continue
                by_pid[pid][k] = by_pid[pid].get(k, 0.0) + float(v)
    return [by_pid[int(pid)] for pid in all_patient_ids]

condition_frequency = build_frequency_dicts(condition_seq, "condition_freq")

lab_frequency_by_level = {
    key: build_frequency_dicts(seq, f"{key}_freq")
    for key, seq in lab_seq_by_level.items()
}

print("Condition frequency patients:", len(condition_frequency))
for key, rows in lab_frequency_by_level.items():
    print(key, "frequency patients:", len(rows))


In [ ]:
# ---------------- Utilization/documentation intensity features ----------------
def build_utilization_dicts(condition_days, lab_days_by_level):
    rows = []
    # Prepare per-patient event summaries.
    cond_tmp = condition_days.copy()
    cond_tmp["n_tokens_day"] = cond_tmp[TOKEN_COL].map(len)

    lab_concat_parts = []
    for key, df in lab_days_by_level.items():
        tmp = df.copy()
        tmp["lab_level"] = key
        tmp["n_tokens_day"] = tmp[TOKEN_COL].map(len)
        lab_concat_parts.append(tmp)
    lab_tmp = pd.concat(lab_concat_parts, ignore_index=True) if lab_concat_parts else pd.DataFrame(columns=[ID_COL, DAY_COL, TOKEN_COL, "n_tokens_day"])

    for pid in all_patient_ids:
        d = {ID_COL: int(pid)}
        cg = cond_tmp[cond_tmp[ID_COL] == pid]
        lg = lab_tmp[lab_tmp[ID_COL] == pid]

        summary = {
            "condition_event_days": cg[DAY_COL].nunique(),
            "condition_total_tokens": cg["n_tokens_day"].sum() if len(cg) else 0,
            "condition_recent_30d": cg.loc[cg[DAY_COL] >= -30, DAY_COL].nunique() if len(cg) else 0,
            "condition_recent_90d": cg.loc[cg[DAY_COL] >= -90, DAY_COL].nunique() if len(cg) else 0,
            "lab_event_days": lg[DAY_COL].nunique(),
            "lab_total_tokens": lg["n_tokens_day"].sum() if len(lg) else 0,
            "lab_recent_30d": lg.loc[lg[DAY_COL] >= -30, DAY_COL].nunique() if len(lg) else 0,
            "lab_recent_90d": lg.loc[lg[DAY_COL] >= -90, DAY_COL].nunique() if len(lg) else 0,
        }
        for k, v in summary.items():
            d[f"util_log1p::{k}"] = math.log1p(float(v))
        rows.append(d)
    return rows

ensure_patient_cohort_initialized()

utilization_features = build_utilization_dicts(condition_days, lab_days_by_level)
print("Utilization feature patients:", len(utilization_features))


In [ ]:
# ---------------- Modeling helpers ----------------
def select_vocab(train_dicts, y_train, top_k=3000, min_patient_freq=5):
    pc = Counter()
    nc = Counter()
    dfc = Counter()
    npos = max(int(np.sum(y_train == 1)), 1)
    nneg = max(int(np.sum(y_train == 0)), 1)
    for d, y in zip(train_dicts, y_train):
        feats = [k for k in d if k != ID_COL]
        for f in feats:
            dfc[f] += 1
            pc[f] += int(y == 1)
            nc[f] += int(y == 0)
    scored = []
    for f, df in dfc.items():
        if df >= min_patient_freq:
            score = abs(pc[f] / npos - nc[f] / nneg) * math.sqrt(df)
            scored.append((score, df, f))
    scored.sort(reverse=True)
    return [f for _, _, f in scored[:top_k]]


def vectorize_dicts(dicts, vocab):
    idx = {f: i for i, f in enumerate(vocab)}
    rows, cols, data = [], [], []
    for r, d in enumerate(dicts):
        for k, v in d.items():
            if k == ID_COL:
                continue
            j = idx.get(k)
            if j is not None:
                rows.append(r)
                cols.append(j)
                data.append(float(v))
    return sparse.csr_matrix((data, (rows, cols)), shape=(len(dicts), len(vocab)), dtype=float)


def normalize_train_test(Xtr, Xte):
    if Xtr.shape[1] == 0:
        return Xtr, Xte
    return normalize(Xtr, norm="l2", axis=1), normalize(Xte, norm="l2", axis=1)


def make_model(family="ridge", y_train=None):
    if family == "ridge":
        return LogisticRegression(
            penalty="l2", solver="liblinear", C=1.0, max_iter=3000,
            class_weight="balanced", random_state=RANDOM_STATE,
        )
    if family == "elasticnet_cv":
        return LogisticRegressionCV(
            Cs=[0.03, 0.1, 0.3, 1.0, 3.0], penalty="elasticnet", solver="saga",
            l1_ratios=[0.05, 0.15, 0.5], cv=3, scoring="roc_auc",
            max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    if family == "xgboost_calibrated":
        if not HAS_XGB:
            return None
        npos = max(int(np.sum(y_train == 1)), 1)
        nneg = max(int(np.sum(y_train == 0)), 1)
        base = XGBClassifier(
            n_estimators=120, max_depth=2, learning_rate=0.03, subsample=0.85,
            colsample_bytree=0.75, reg_lambda=4.0, reg_alpha=0.5, min_child_weight=8,
            objective="binary:logistic", eval_metric="logloss", scale_pos_weight=nneg / npos,
            random_state=RANDOM_STATE, n_jobs=4,
        )
        return CalibratedClassifierCV(base, method="sigmoid", cv=3)
    if family == "lightgbm_calibrated":
        if not HAS_LGBM:
            return None
        npos = max(int(np.sum(y_train == 1)), 1)
        nneg = max(int(np.sum(y_train == 0)), 1)
        base = lgb.LGBMClassifier(
            objective="binary", n_estimators=160, learning_rate=0.03,
            num_leaves=7, max_depth=3, min_child_samples=40,
            subsample=0.85, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=5.0,
            scale_pos_weight=nneg / npos, random_state=RANDOM_STATE, n_jobs=4, verbose=-1,
        )
        return CalibratedClassifierCV(base, method="sigmoid", cv=3)
    raise ValueError(f"Unknown model family: {family}")


def threshold_at_specificity(y, prob, target):
    best = None
    for thr in np.unique(prob):
        pred = (prob >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
        spec = tn / max(tn + fp, 1)
        if spec >= target:
            sens = tp / max(tp + fn, 1)
            ppv = tp / max(tp + fp, 1) if (tp + fp) > 0 else 0.0
            cand = (thr, sens, spec, ppv)
            if best is None or cand[1] > best[1]:
                best = cand
    if best is None:
        return np.nan, np.nan, np.nan, np.nan
    return best


def compute_metrics(y, prob, name):
    m = {
        "model": name,
        "n": int(len(y)),
        "case_rate": float(np.mean(y)),
        "auroc": float(roc_auc_score(y, prob)),
        "auprc": float(average_precision_score(y, prob)),
        "brier": float(brier_score_loss(y, prob)),
    }
    for target in SPEC_TARGETS:
        thr, sens, spec, ppv = threshold_at_specificity(y, prob, target)
        tag = f"spec{int(target * 100)}"
        m[f"{tag}_threshold"] = thr
        m[f"{tag}_sensitivity"] = sens
        m[f"{tag}_specificity"] = spec
        m[f"{tag}_ppv"] = ppv
    return m


def run_feature_model(name, dicts, family="ridge", top_k=3000):
    ids = y_df[ID_COL].astype(int).values
    y = y_df[LABEL_COL].astype(int).values
    by = {int(d[ID_COL]): d for d in dicts}
    aligned = [by.get(int(pid), {ID_COL: int(pid)}) for pid in ids]

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.full(len(ids), np.nan)
    coef_rows = []
    fold_rows = []

    for fold, (tr, te) in enumerate(skf.split(ids, y), 1):
        vocab = select_vocab([aligned[i] for i in tr], y[tr], top_k=top_k, min_patient_freq=MIN_PATIENT_FREQ)
        Xtr = vectorize_dicts([aligned[i] for i in tr], vocab)
        Xte = vectorize_dicts([aligned[i] for i in te], vocab)
        Xtr, Xte = normalize_train_test(Xtr, Xte)

        if Xtr.shape[1] == 0:
            oof[te] = np.mean(y[tr])
            fold_rows.append({"model": name, "fold": fold, "n_features": 0})
            continue

        model = make_model(family=family, y_train=y[tr])
        if model is None:
            oof[te] = np.mean(y[tr])
            fold_rows.append({"model": name, "fold": fold, "n_features": Xtr.shape[1], "skipped": True})
            continue

        model.fit(Xtr, y[tr])
        prob = model.predict_proba(Xte)[:, 1]
        oof[te] = prob
        fold_rows.append({"model": name, "fold": fold, "n_features": Xtr.shape[1], "fold_auroc": roc_auc_score(y[te], prob)})
        print(name, "fold", fold, "features", Xtr.shape[1], "AUROC", round(roc_auc_score(y[te], prob), 3))

        # Feature importance for linear models only. Tree importances are less direct after calibration.
        estimator = model
        if hasattr(model, "base_estimator"):
            estimator = model.base_estimator
        if hasattr(estimator, "coef_"):
            coefs = estimator.coef_.ravel()
            for f, c in zip(vocab, coefs):
                if abs(c) > 1e-12:
                    coef_rows.append({"model": name, "fold": fold, "feature": f, "coef": float(c), "abs_coef": float(abs(c))})

    metrics = compute_metrics(y, oof, name)
    metrics["prediction_head"] = family
    metrics["n_features_median_fold"] = float(np.median([r.get("n_features", 0) for r in fold_rows]))

    pred_df = pd.DataFrame({ID_COL: ids, LABEL_COL: y, f"p_{name}": oof})
    return metrics, pred_df, pd.DataFrame(coef_rows), pd.DataFrame(fold_rows)


In [ ]:
# ---------------- Defensive pre-model feature-block initialization ----------------
# This cell makes the notebook robust if cells are rerun out of order.
# It rebuilds patient cohort, sequences, frequency features, and utilization features
# immediately before model specification construction.

ensure_patient_cohort_initialized()


def table_to_sequences(event_df):
    seqs = {}
    if event_df is None or len(event_df) == 0:
        return seqs
    for pid, g in event_df.groupby(ID_COL):
        rows = []
        for _, r in g.sort_values(DAY_COL).iterrows():
            toks_raw = r[TOKEN_COL]
            if isinstance(toks_raw, str):
                toks = parse_token_payload(toks_raw) if 'parse_token_payload' in globals() else [toks_raw]
            else:
                toks = toks_raw
            toks = sorted(set(str(t).strip() for t in toks if str(t).strip()))
            if toks:
                rows.append((int(r[DAY_COL]), toks))
        seqs[int(pid)] = rows
    return seqs


def build_frequency_dicts(seqs, prefix):
    rows = []
    for pid in all_patient_ids:
        d = {ID_COL: int(pid)}
        cnt = Counter()
        for day, toks in seqs.get(int(pid), []):
            for tok in toks:
                cnt[f"{prefix}::{tok}"] += 1
        for k, v in cnt.items():
            d[k] = math.log1p(min(v, 20))
        rows.append(d)
    return rows


def build_cross_transition_dicts(
    source_seqs,
    target_seqs,
    source_prefix,
    target_prefix,
    direction_name,
    max_lag,
    tau,
    mode="forward",
    include_same_day=False,
):
    """Build cross-modal condition-lab transition features.

    mode='forward': source event must occur before target event.
    mode='bidirectional': source and target are paired if abs(day gap) <= max_lag, ignoring order.
    """
    rows = []
    for pid in all_patient_ids:
        d = {ID_COL: int(pid)}
        src_events = [(day, tok) for day, toks in source_seqs.get(int(pid), []) for tok in toks]
        tgt_events = [(day, tok) for day, toks in target_seqs.get(int(pid), []) for tok in toks]
        src_events = sorted(src_events)
        tgt_events = sorted(tgt_events)
        trans = Counter()

        if mode == "forward":
            for d1, a in src_events:
                for d2, b in tgt_events:
                    gap = d2 - d1
                    if gap < 0:
                        continue
                    if gap == 0 and not include_same_day:
                        continue
                    if gap > max_lag:
                        continue
                    w = math.exp(-gap / max(float(tau), 1e-9))
                    key = f"{direction_name}_forward_lag{max_lag}_tau{tau}::{source_prefix}:{a}->{target_prefix}:{b}"
                    trans[key] += w

        elif mode == "bidirectional":
            for d1, a in src_events:
                for d2, b in tgt_events:
                    gap = abs(d2 - d1)
                    if gap == 0 and not include_same_day:
                        continue
                    if gap > max_lag:
                        continue
                    w = math.exp(-gap / max(float(tau), 1e-9))
                    key = f"{direction_name}_bidirectional_lag{max_lag}_tau{tau}::{source_prefix}:{a}<->{target_prefix}:{b}"
                    trans[key] += w
        else:
            raise ValueError(f"Unknown mode: {mode}")

        for k, v in trans.items():
            d[k] = v
        rows.append(d)
    return rows


def merge_dicts(name, dict_lists):
    by_pid = {int(pid): {ID_COL: int(pid)} for pid in all_patient_ids}
    for dicts in dict_lists:
        for d in dicts:
            pid = int(d[ID_COL])
            if pid not in by_pid:
                continue
            for k, v in d.items():
                if k == ID_COL:
                    continue
                by_pid[pid][k] = by_pid[pid].get(k, 0.0) + float(v)
    return [by_pid[int(pid)] for pid in all_patient_ids]


def build_utilization_dicts(condition_days, lab_days_by_level):
    rows = []
    cond_tmp = condition_days.copy()
    cond_tmp["n_tokens_day"] = cond_tmp[TOKEN_COL].map(len)

    lab_concat_parts = []
    for key, df in lab_days_by_level.items():
        tmp = df.copy()
        tmp["lab_level"] = key
        tmp["n_tokens_day"] = tmp[TOKEN_COL].map(len)
        lab_concat_parts.append(tmp)
    lab_tmp = pd.concat(lab_concat_parts, ignore_index=True) if lab_concat_parts else pd.DataFrame(columns=[ID_COL, DAY_COL, TOKEN_COL, "n_tokens_day"])

    for pid in all_patient_ids:
        d = {ID_COL: int(pid)}
        cg = cond_tmp[cond_tmp[ID_COL] == pid]
        lg = lab_tmp[lab_tmp[ID_COL] == pid]
        summary = {
            "condition_event_days": cg[DAY_COL].nunique(),
            "condition_total_tokens": cg["n_tokens_day"].sum() if len(cg) else 0,
            "condition_recent_30d": cg.loc[cg[DAY_COL] >= -30, DAY_COL].nunique() if len(cg) else 0,
            "condition_recent_90d": cg.loc[cg[DAY_COL] >= -90, DAY_COL].nunique() if len(cg) else 0,
            "lab_event_days": lg[DAY_COL].nunique(),
            "lab_total_tokens": lg["n_tokens_day"].sum() if len(lg) else 0,
            "lab_recent_30d": lg.loc[lg[DAY_COL] >= -30, DAY_COL].nunique() if len(lg) else 0,
            "lab_recent_90d": lg.loc[lg[DAY_COL] >= -90, DAY_COL].nunique() if len(lg) else 0,
        }
        for k, v in summary.items():
            d[f"util_log1p::{k}"] = math.log1p(float(v))
        rows.append(d)
    return rows

# Rebuild all feature blocks from current event tables.
condition_seq = table_to_sequences(condition_days)
lab_seq_by_level = {key: table_to_sequences(df) for key, df in lab_days_by_level.items()}
condition_frequency = build_frequency_dicts(condition_seq, "condition_freq")
lab_frequency_by_level = {key: build_frequency_dicts(seq, f"{key}_freq") for key, seq in lab_seq_by_level.items()}
utilization_features = build_utilization_dicts(condition_days, lab_days_by_level)

print("Pre-model feature initialization complete.")
print("Patients:", len(all_patient_ids))
print("Condition frequency rows:", len(condition_frequency))
for key, rows in lab_frequency_by_level.items():
    print(key, "frequency rows:", len(rows), "sequence patients:", len(lab_seq_by_level.get(key, {})))
print("Utilization rows:", len(utilization_features))


## Cross-modal directionality encodings

This notebook now evaluates four condition-laboratory transition encodings:

1. `condition_to_lab_forward`: condition event precedes abnormal lab state. This is the primary physiologic manifestation / monitoring pathway.
2. `lab_to_condition_forward`: abnormal lab state precedes condition event. This is the reverse directed diagnostic/documentation-response pathway.
3. `condition_lab_directed_both`: both directed feature sets are included in the same model (`condition -> lab` and `lab -> condition`), while preserving their separate directions. This tests a directed cross-modal feedback loop.
4. `condition_lab_proximity_bidirectional`: symmetric proximity feature (`condition <-> lab`) where temporal order is ignored. This is a co-documentation/proximity sensitivity analysis and should not be interpreted as causal sequence.

The `directed_both` model is different from bidirectional proximity: it preserves chronological direction by keeping `condition -> lab` and `lab -> condition` as distinct features.


In [ ]:
# ---------------- Build model specifications ----------------
# Baselines.
MODEL_SPECS = []
MODEL_SPECS.append(("condition_frequency_ridge", condition_frequency, "ridge", TOP_K_FREQ))
MODEL_SPECS.append(("utilization_only_ridge", utilization_features, "ridge", 100))

for lab_key, lab_freq in lab_frequency_by_level.items():
    MODEL_SPECS.append((f"{lab_key}_frequency_ridge", lab_freq, "ridge", TOP_K_FREQ))
    MODEL_SPECS.append((f"condition_plus_{lab_key}_frequency_ridge", merge_dicts("tmp", [condition_frequency, lab_freq]), "ridge", TOP_K_COMBINED))
    MODEL_SPECS.append((f"condition_plus_{lab_key}_frequency_plus_utilization_ridge", merge_dicts("tmp", [condition_frequency, lab_freq, utilization_features]), "ridge", TOP_K_COMBINED))

# Cross-transition specs.
for lab_key, lab_seq in lab_seq_by_level.items():
    lab_freq = lab_frequency_by_level[lab_key]
    for lag, tau in LAG_TAU_GRID:
        # Primary ordered condition -> lab.
        cl_forward = build_cross_transition_dicts(
            condition_seq, lab_seq,
            source_prefix="condition", target_prefix=lab_key,
            direction_name=f"condition_to_{lab_key}",
            max_lag=lag, tau=tau, mode="forward", include_same_day=False,
        )
        MODEL_SPECS.append((f"condition_to_{lab_key}_forward_lag{lag}_tau{tau}_ridge", cl_forward, "ridge", TOP_K_TRANSITION))
        MODEL_SPECS.append((f"condition_freq_plus_condition_to_{lab_key}_forward_lag{lag}_tau{tau}_ridge", merge_dicts("tmp", [condition_frequency, cl_forward]), "ridge", TOP_K_COMBINED))
        MODEL_SPECS.append((f"condition_lab_freq_plus_condition_to_{lab_key}_forward_lag{lag}_tau{tau}_ridge", merge_dicts("tmp", [condition_frequency, lab_freq, cl_forward]), "ridge", TOP_K_COMBINED))
        MODEL_SPECS.append((f"condition_lab_freq_plus_condition_to_{lab_key}_forward_lag{lag}_tau{tau}_plus_utilization_ridge", merge_dicts("tmp", [condition_frequency, lab_freq, cl_forward, utilization_features]), "ridge", TOP_K_COMBINED))

        # Reverse ordered lab -> condition.
        lc_forward = build_cross_transition_dicts(
            lab_seq, condition_seq,
            source_prefix=lab_key, target_prefix="condition",
            direction_name=f"{lab_key}_to_condition",
            max_lag=lag, tau=tau, mode="forward", include_same_day=False,
        )
        MODEL_SPECS.append((f"{lab_key}_to_condition_forward_lag{lag}_tau{tau}_ridge", lc_forward, "ridge", TOP_K_TRANSITION))
        MODEL_SPECS.append((f"condition_lab_freq_plus_{lab_key}_to_condition_forward_lag{lag}_tau{tau}_ridge", merge_dicts("tmp", [condition_frequency, lab_freq, lc_forward]), "ridge", TOP_K_COMBINED))
        MODEL_SPECS.append((f"condition_lab_freq_plus_{lab_key}_to_condition_forward_lag{lag}_tau{tau}_plus_utilization_ridge", merge_dicts("tmp", [condition_frequency, lab_freq, lc_forward, utilization_features]), "ridge", TOP_K_COMBINED))

        # Directed-both sensitivity: include both condition -> lab and lab -> condition, preserving direction.
        # This is NOT the same as symmetric bidirectional proximity.
        directed_both = merge_dicts("tmp", [cl_forward, lc_forward])
        MODEL_SPECS.append((f"condition_{lab_key}_directed_both_lag{lag}_tau{tau}_ridge", directed_both, "ridge", TOP_K_TRANSITION))
        MODEL_SPECS.append((f"condition_freq_plus_condition_{lab_key}_directed_both_lag{lag}_tau{tau}_ridge", merge_dicts("tmp", [condition_frequency, directed_both]), "ridge", TOP_K_COMBINED))
        MODEL_SPECS.append((f"condition_lab_freq_plus_condition_{lab_key}_directed_both_lag{lag}_tau{tau}_ridge", merge_dicts("tmp", [condition_frequency, lab_freq, directed_both]), "ridge", TOP_K_COMBINED))
        MODEL_SPECS.append((f"condition_lab_freq_plus_condition_{lab_key}_directed_both_lag{lag}_tau{tau}_plus_utilization_ridge", merge_dicts("tmp", [condition_frequency, lab_freq, directed_both, utilization_features]), "ridge", TOP_K_COMBINED))

        # Bidirectional proximity sensitivity. Order is ignored; not causal/ordered.
        cl_bidir = build_cross_transition_dicts(
            condition_seq, lab_seq,
            source_prefix="condition", target_prefix=lab_key,
            direction_name=f"condition_{lab_key}_proximity",
            max_lag=lag, tau=tau, mode="bidirectional", include_same_day=False,
        )
        MODEL_SPECS.append((f"condition_lab_freq_plus_condition_{lab_key}_proximity_bidirectional_lag{lag}_tau{tau}_ridge", merge_dicts("tmp", [condition_frequency, lab_freq, cl_bidir]), "ridge", TOP_K_COMBINED))
        MODEL_SPECS.append((f"condition_lab_freq_plus_condition_{lab_key}_proximity_bidirectional_lag{lag}_tau{tau}_plus_utilization_ridge", merge_dicts("tmp", [condition_frequency, lab_freq, cl_bidir, utilization_features]), "ridge", TOP_K_COMBINED))

print("Model specs to run:", len(MODEL_SPECS))
for i, (name, _, family, _) in enumerate(MODEL_SPECS[:25], 1):
    print(f"[{i}/{len(MODEL_SPECS)}]", name)
if len(MODEL_SPECS) > 25:
    print("...")


In [ ]:
# ---------------- Run lag/tau models ----------------
metrics_rows = []
pred_tables = []
coef_tables = []
fold_tables = []

for i, (name, dicts, family, top_k) in enumerate(MODEL_SPECS, 1):
    print(f"[{i}/{len(MODEL_SPECS)}] {name}")
    try:
        m, pred, coef, fold_info = run_feature_model(name, dicts, family=family, top_k=top_k)
        metrics_rows.append(m)
        pred_tables.append(pred)
        if len(coef):
            coef_tables.append(coef)
        if len(fold_info):
            fold_tables.append(fold_info)
    except Exception as e:
        print("FAILED:", name, e)
        metrics_rows.append({"model": name, "error": str(e), "prediction_head": family})

metrics_df = pd.DataFrame(metrics_rows)
metrics_path = OUT_DIR / "condition_lab_lag_tau_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
print("Saved:", metrics_path)

# Patient-level OOF predictions contain study IDs and are not written by default in the public-safe notebook.
if SAVE_PATIENT_LEVEL_OUTPUTS:
    oof = y_df[[ID_COL, LABEL_COL]].copy()
    for p in pred_tables:
        prob_cols = [c for c in p.columns if c.startswith("p_")]
        oof = oof.merge(p[[ID_COL] + prob_cols], on=ID_COL, how="left")
    oof_path = OUT_DIR / "condition_lab_lag_tau_oof_predictions_DO_NOT_COMMIT.csv"
    oof.to_csv(oof_path, index=False)
    print("Saved patient-level OOF predictions:", oof_path)
else:
    print("Skipped patient-level OOF prediction export. Set SAVE_PATIENT_LEVEL_OUTPUTS=True only for approved local analysis.")

coef_df = pd.concat(coef_tables, ignore_index=True) if coef_tables else pd.DataFrame()
if SAVE_COEFFICIENTS:
    coef_path = OUT_DIR / "condition_lab_lag_tau_coefficients.csv"
    coef_df.to_csv(coef_path, index=False)
    print("Saved:", coef_path)

fold_df = pd.concat(fold_tables, ignore_index=True) if fold_tables else pd.DataFrame()
if SAVE_FOLD_INFO:
    fold_path = OUT_DIR / "condition_lab_lag_tau_fold_info.csv"
    fold_df.to_csv(fold_path, index=False)
    print("Saved:", fold_path)

display(metrics_df.sort_values("auroc", ascending=False, na_position="last").head(30))


In [ ]:
# ---------------- Lag/tau heatmaps and directional summaries ----------------
def parse_model_name_for_grid(name):
    s = str(name)
    lag_m = re.search(r"lag(\d+)", s)
    tau_m = re.search(r"tau(\d+)", s)
    if not lag_m or not tau_m:
        return None
    lag = int(lag_m.group(1))
    tau = int(tau_m.group(1))
    if "directed_both" in s:
        mode = "directed_both"
    elif "bidirectional" in s or "proximity" in s:
        mode = "bidirectional_proximity"
    elif "to_condition_forward" in s:
        mode = "lab_to_condition_forward"
    elif "condition_to" in s and "forward" in s:
        mode = "condition_to_lab_forward"
    else:
        mode = "other"
    if "labaxis" in s:
        lab_level = "labaxis"
    elif "labstate" in s:
        lab_level = "labstate"
    else:
        lab_level = "unknown"
    return {"lag": lag, "tau": tau, "mode": mode, "lab_level": lab_level}

plot_df = metrics_df.copy()
parsed = plot_df["model"].apply(parse_model_name_for_grid)
parsed_df = pd.DataFrame([p if p is not None else {} for p in parsed])
plot_df = pd.concat([plot_df, parsed_df], axis=1)

summary_cols = ["mode", "lab_level", "model", "n", "auroc", "auprc", "brier", "n_features_median_fold"]
print("Best model within each direction/level:")
display(
    plot_df.dropna(subset=["auroc"])
    .sort_values("auroc", ascending=False)
    .groupby(["mode", "lab_level"], dropna=False)
    .head(1)[summary_cols]
    .sort_values(["mode", "lab_level"])
)

for mode in ["condition_to_lab_forward", "lab_to_condition_forward", "directed_both", "bidirectional_proximity"]:
    for lab_level in LAB_LEVELS:
        sub = plot_df[(plot_df["mode"] == mode) & (plot_df["lab_level"] == lab_level)].copy()
        # Focus on combined frequency + transition models because these are most comparable.
        sub = sub[sub["model"].str.contains("condition_lab_freq_plus", regex=False, na=False)]
        if sub.empty:
            continue
        piv = sub.pivot_table(index="lag", columns="tau", values="auroc", aggfunc="max")
        print("\n", mode, lab_level)
        display(piv.round(4))
        fig, ax = plt.subplots(figsize=(6, 4))
        im = ax.imshow(piv.values, aspect="auto")
        ax.set_xticks(range(len(piv.columns)))
        ax.set_xticklabels(piv.columns)
        ax.set_yticks(range(len(piv.index)))
        ax.set_yticklabels(piv.index)
        ax.set_xlabel("tau")
        ax.set_ylabel("lag")
        ax.set_title(f"{mode} | {lab_level}")
        plt.colorbar(im, ax=ax, label="AUROC")
        plt.show()


In [ ]:
# ---------------- Optional prediction-head tuning on frozen best blocks ----------------
# This is intentionally downstream from lag/tau tuning. It should not be used to choose lag/tau.
ENABLE_PREDICTION_HEAD_TUNING = RUN_PREDICTION_HEAD_TUNING
PREDICTION_HEADS = ["ridge", "elasticnet_cv", "xgboost_calibrated", "lightgbm_calibrated"]

if ENABLE_PREDICTION_HEAD_TUNING:
    ranked = metrics_df.dropna(subset=["auroc"]).sort_values("auroc", ascending=False).copy()
    frozen_specs = []

    # Always include key baselines.
    baseline_names = [
        "condition_frequency_ridge",
        "utilization_only_ridge",
    ]
    for name in baseline_names:
        hit = [spec for spec in MODEL_SPECS if spec[0] == name]
        if hit:
            frozen_specs.append(hit[0])

    # Add best condition->lab, lab->condition, directed-both, and bidirectional proximity models.
    for mode_pat in ["condition_to", "to_condition_forward", "directed_both", "proximity_bidirectional"]:
        sub = ranked[ranked["model"].str.contains(mode_pat, na=False)].head(1)
        for _, r in sub.iterrows():
            hit = [spec for spec in MODEL_SPECS if spec[0] == r["model"]]
            if hit:
                frozen_specs.append(hit[0])

    # Deduplicate.
    seen = set()
    frozen_specs_unique = []
    for spec in frozen_specs:
        if spec[0] not in seen:
            frozen_specs_unique.append(spec)
            seen.add(spec[0])
    frozen_specs = frozen_specs_unique

    print("Frozen feature blocks for prediction-head tuning:", len(frozen_specs))
    for s in frozen_specs:
        print(" -", s[0])

    head_metrics = []
    head_preds = []
    head_coefs = []
    head_folds = []
    for block_name, dicts, _, top_k in frozen_specs:
        for head in PREDICTION_HEADS:
            if head == "xgboost_calibrated" and not HAS_XGB:
                continue
            if head == "lightgbm_calibrated" and not HAS_LGBM:
                continue
            name = f"{block_name}__{head}"
            print("Tuning head:", name)
            try:
                m, pred, coef, fold_info = run_feature_model(name, dicts, family=head, top_k=top_k)
                m["frozen_block"] = block_name
                head_metrics.append(m)
                head_preds.append(pred)
                if len(coef):
                    head_coefs.append(coef)
                if len(fold_info):
                    head_folds.append(fold_info)
            except Exception as e:
                print("FAILED:", name, e)
                head_metrics.append({"model": name, "frozen_block": block_name, "prediction_head": head, "error": str(e)})

    head_metrics_df = pd.DataFrame(head_metrics)
    head_metrics_path = OUT_DIR / "condition_lab_prediction_head_tuning_metrics.csv"
    head_metrics_df.to_csv(head_metrics_path, index=False)
    print("Saved:", head_metrics_path)

    if SAVE_PATIENT_LEVEL_OUTPUTS:
        head_oof = y_df[[ID_COL, LABEL_COL]].copy()
        for p in head_preds:
            prob_cols = [c for c in p.columns if c.startswith("p_")]
            head_oof = head_oof.merge(p[[ID_COL] + prob_cols], on=ID_COL, how="left")
        head_oof_path = OUT_DIR / "condition_lab_prediction_head_tuning_oof_predictions_DO_NOT_COMMIT.csv"
        head_oof.to_csv(head_oof_path, index=False)
        print("Saved patient-level prediction-head OOF predictions:", head_oof_path)
    else:
        print("Skipped patient-level prediction-head OOF export. Set SAVE_PATIENT_LEVEL_OUTPUTS=True only for approved local analysis.")

    display(head_metrics_df.sort_values("auroc", ascending=False, na_position="last").head(20))
else:
    print("Prediction-head tuning disabled.")


In [ ]:
# ---------------- VVUQ manifest ----------------
manifest = {
    "notebook": "condition_lab_stdp_lag_tau_directed_both_public_safe",
    "created_for": "public-safe methodological notebook",
    "cohort_n": int(y_df[ID_COL].nunique()),
    "case_rate": float(y_df[LABEL_COL].mean()),
    "transition_modes": {
        "primary": PRIMARY_TRANSITION_MODES,
        "sensitivity": SENSITIVITY_TRANSITION_MODES,
    },
    "lag_tau_grid": LAG_TAU_GRID,
    "lab_levels": LAB_LEVELS,
    "lab_state_filters": LAB_STATE_FILTERS,
    "interpretation": {
        "forward": "temporally ordered association; suitable for trajectory sensitivity but not causal by itself",
        "directed_both": "both condition-to-lab and lab-to-condition directed feature sets included in one model while preserving direction",
        "bidirectional": "symmetric temporal proximity/co-documentation; not causal or disease-progression edge",
        "lag": "maximum allowed event-pair gap in days",
        "tau": "exponential decay parameter in days",
    },
    "outputs": [
        "condition_lab_lag_tau_metrics.csv",
        "condition_lab_lag_tau_coefficients.csv",
        "condition_lab_lag_tau_fold_info.csv",
        "condition_lab_prediction_head_tuning_metrics.csv",
    ],
    "patient_level_outputs_default": bool(SAVE_PATIENT_LEVEL_OUTPUTS),
    "privacy_note": "Do not commit raw data, patient-level predictions, direct identifiers, or files containing study IDs unless synthetic or approved for release.",
}
manifest_path = OUT_DIR / "condition_lab_stdp_vvuq_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print("Saved:", manifest_path)
print(json.dumps(manifest, indent=2)[:2000])
